In [29]:
import os
import sys
sys.path.append(os.path.abspath('..'))

from src.models import stacking_ensemble

import pandas as pd
import numpy as np

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
lgbm_oof_proba = np.load('oof_probas/oof_LGBM_probas.npy')
xgb_oof_proba  = np.load('oof_probas/oof_XGB_probas.npy')
et_oof_proba   = np.load('oof_probas/oof_ETC_probas.npy')
nn_oof_proba   = np.load('oof_probas/oof_nn_probas.npy')
realmlp_oof_proba = np.load('oof_probas/oof_realmlp_probas.npy')
# tabm_oof_proba     = np.load('oof_probas/oof_tabm_probas.npy')

lgbm_test_proba = np.load('test_probas/LGBM_test_probas.npy')
xgb_test_proba  = np.load('test_probas/XGB_test_probas.npy')
et_test_proba   = np.load('test_probas/ETC_test_probas.npy')
nn_test_proba   = np.load('test_probas/nn_test_probas.npy')
realmlp_test_proba = np.load('test_probas/realmlp_test_proba.npy')
# tabm_test_proba     = np.load('test_probas/tabm_test_proba.npy')

In [ ]:
from sklearn.linear_model import LogisticRegression

meta_model = LogisticRegression(
    max_iter=3000,
    C=0.1,     
    random_state=42
)

In [ ]:
y_resa = pd.read_csv(r"y_resa.csv")
test_df = pd.read_csv(r"..\data\raw\test.csv")

preds,probas,model = stacking_ensemble(oof_probas_list=[lgbm_oof_proba,xgb_oof_proba,et_oof_proba,nn_oof_proba,realmlp_oof_proba],                                       
                                        y=y_resa,
                                        test_probas_list=[lgbm_test_proba,xgb_test_proba,et_test_proba,nn_test_proba,realmlp_test_proba],
                                        meta_model=meta_model)


class_map = {0: 'GALAXY', 1: 'QSO', 2: 'STAR'}
pred_labels = [class_map[p] for p in preds]

submission = pd.DataFrame({
    'id':    test_df['id'],
    'class': pred_labels
})

submission.to_csv('submissions/stacking_submission.csv', index=False)

print("Submission saved successfully!")